In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [ ]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.LM(
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=False,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="solve",
    batch_size=100,
    # batch_size=None,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.40096303820610046
epoch:  1, loss: 0.3441154658794403
epoch:  2, loss: 0.26931968331336975
epoch:  3, loss: 0.1951029896736145
epoch:  4, loss: 0.12538142502307892
epoch:  5, loss: 0.06848149746656418
epoch:  6, loss: 0.03632472828030586
epoch:  7, loss: 0.031987909227609634
epoch:  8, loss: 0.031185725703835487
epoch:  9, loss: 0.023085379973053932
epoch:  10, loss: 0.02145819552242756
epoch:  11, loss: 0.01992115005850792
epoch:  12, loss: 0.01907501183450222
epoch:  13, loss: 0.01800532452762127
epoch:  14, loss: 0.017252160236239433
epoch:  15, loss: 0.016864342615008354
epoch:  16, loss: 0.015568837523460388
epoch:  17, loss: 0.014789368957281113
epoch:  18, loss: 0.014314491301774979
epoch:  19, loss: 0.013995679095387459
epoch:  20, loss: 0.013327054679393768
epoch:  21, loss: 0.012722999788820744
epoch:  22, loss: 0.012397718615829945
epoch:  23, loss: 0.012209270149469376
epoch:  24, loss: 0.01201589684933424
epoch:  25, loss: 0.01161674503237009
epoch:  26,

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9286116361618042
Test metrics:  R2 = 0.9352073073387146


In [10]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = torch_numopt.LM(
    model=model,
    lr=1,
    mu=0.001,
    mu_dec=0.1,
    fletcher=True,
    c1=1e-4,
    tau=0.1,
    line_search_method="backtrack",
    line_search_cond="armijo",
    solver="pinv",
    batch_size=100,
    # batch_size=None,
)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3413708508014679
epoch:  1, loss: 0.255596399307251
epoch:  2, loss: 0.18057483434677124
epoch:  3, loss: 0.10557010769844055
epoch:  4, loss: 0.05034197121858597
epoch:  5, loss: 0.03211700916290283
epoch:  6, loss: 0.031445033848285675
epoch:  7, loss: 0.03129878267645836
epoch:  8, loss: 0.031226933002471924
epoch:  9, loss: 0.031176332384347916
epoch:  10, loss: 0.031137457117438316
epoch:  11, loss: 0.031100379303097725
epoch:  12, loss: 0.031031761318445206
epoch:  13, loss: 0.030820373445749283
epoch:  14, loss: 0.030671648681163788
epoch:  15, loss: 0.030618958175182343
epoch:  16, loss: 0.027805136516690254
epoch:  17, loss: 0.024545278400182724
epoch:  18, loss: 0.018462741747498512
epoch:  19, loss: 0.017747465521097183
epoch:  20, loss: 0.017117690294981003
epoch:  21, loss: 0.014047665521502495
epoch:  22, loss: 0.013195282779633999
epoch:  23, loss: 0.012873459607362747
epoch:  24, loss: 0.011697366833686829
epoch:  25, loss: 0.011149573139846325
epoch:

In [11]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.7616668939590454
Test metrics:  R2 = 0.7419722676277161
